# Healthcare Disease-Progression Early Warning

**Predict one-year diabetes disease progression using general clinical measurements only.**

### Clinical / analytical question
Can baseline physiological and serum measurements identify patients with elevated future disease-progression risk?

### Real dataset
This notebook uses scikit-learn's diabetes progression dataset:
- **442 patients**
- quantitative disease-progression outcome measured **one year after baseline**
- baseline BMI, average blood pressure and six serum measurements

### Privacy / feature policy
The original dataset also contains **age and sex**, but this project deliberately removes them before modelling.

**Model inputs:** BMI, blood pressure, total serum cholesterol, LDL, HDL, total-cholesterol/HDL ratio, serum triglyceride-related signal and blood glucose.

**Not used:** age, sex, names, patient IDs or other demographics.

### System design
- train / validation / untouched test split
- Ridge, Random Forest and HistGradientBoosting regression
- validation-selected champion model
- MAE, RMSE and R²
- distribution-free residual prediction intervals
- separate high-progression risk classifier
- ROC-AUC, PR-AUC, precision / recall / F1
- permutation importance
- ranked anonymised clinical-review queue

> **Educational decision-support project only. It is not a diagnostic tool or medical advice.**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report,
    precision_recall_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Environment ready.")

## 1. Load the real diabetes progression dataset

The response is a quantitative measure of disease progression one year after the baseline examination.

In [ ]:
dataset = load_diabetes(as_frame=True, scaled=True)
X_full = dataset.data.copy()
y = dataset.target.astype(float).copy()

print(f"Patients: {len(X_full):,}")
print(f"Original features: {X_full.columns.tolist()}")
print(f"Target range: {y.min():.0f} -> {y.max():.0f}")
print(f"Target mean: {y.mean():.2f}")
display(X_full.head())

## 2. Remove demographic variables completely

Age and sex are removed before model fitting, scaling, model selection, thresholding or explanation.

In [ ]:
DROP_DEMOGRAPHICS = ["age", "sex"]
X = X_full.drop(columns=DROP_DEMOGRAPHICS).copy()

feature_labels = {
    "bmi": "BMI",
    "bp": "Average blood pressure",
    "s1": "Total serum cholesterol",
    "s2": "LDL",
    "s3": "HDL",
    "s4": "Total cholesterol / HDL",
    "s5": "Serum triglyceride-related signal",
    "s6": "Blood glucose"
}
X = X.rename(columns=feature_labels)

print("Features retained:")
for c in X.columns:
    print(" -", c)
print("\nAge present:", "age" in X.columns)
print("Sex present:", "sex" in X.columns)
print("Final shape:", X.shape)

## 3. Exploratory clinical signal analysis

In [ ]:
target_summary = pd.Series({
    "patients": len(y),
    "mean_progression": y.mean(),
    "median_progression": y.median(),
    "std_progression": y.std(),
    "min_progression": y.min(),
    "max_progression": y.max()
})
display(target_summary.to_frame("value"))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(y, bins=25)
ax.set_title("One-year disease progression target distribution")
ax.set_xlabel("Progression measure")
ax.set_ylabel("Patients")
plt.tight_layout()
plt.show()

In [ ]:
correlations = X.assign(target=y.values).corr(numeric_only=True)["target"].drop("target").sort_values(key=np.abs, ascending=False)
display(correlations.to_frame("correlation_with_progression"))

fig, ax = plt.subplots(figsize=(8, 5))
correlations.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Baseline clinical signals vs one-year progression")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 4. Leakage-safe train / validation / test split

Model selection and interval width are determined using validation data only. The final test set remains untouched until the end.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "patients": [len(X_train), len(X_val), len(X_test)],
    "target_mean": [y_train.mean(), y_val.mean(), y_test.mean()],
    "target_std": [y_train.std(), y_val.std(), y_test.std()]
}, index=["train","validation","test"])
display(split_summary)

## 5. Regression models — predict future disease progression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train_scaled, y_train)

rf = RandomForestRegressor(
    n_estimators=600,
    min_samples_leaf=4,
    max_features=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=350,
    max_leaf_nodes=15,
    l2_regularization=2.0,
    random_state=RANDOM_STATE
)
hgb.fit(X_train, y_train)

val_pred_ridge = ridge.predict(X_val_scaled)
val_pred_rf = rf.predict(X_val)
val_pred_hgb = hgb.predict(X_val)

print("Models trained.")

In [ ]:
def regression_metrics(name, y_true, pred):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred)
    }

validation_results = pd.DataFrame([
    regression_metrics("Ridge", y_val, val_pred_ridge),
    regression_metrics("Random Forest", y_val, val_pred_rf),
    regression_metrics("HistGradientBoosting", y_val, val_pred_hgb)
]).set_index("model").sort_values("MAE")

display(validation_results.style.format({
    "MAE":"{:.2f}",
    "RMSE":"{:.2f}",
    "R2":"{:.4f}"
}))

best_regression_name = validation_results.index[0]
print("Validation-selected regression model:", best_regression_name)

## 6. Distribution-free uncertainty interval

Absolute residuals on the validation set estimate a 90% prediction-error band. The interval is frozen before final test evaluation.

In [ ]:
val_prediction_map = {
    "Ridge": val_pred_ridge,
    "Random Forest": val_pred_rf,
    "HistGradientBoosting": val_pred_hgb
}
selected_val_pred = val_prediction_map[best_regression_name]

absolute_residuals = np.abs(y_val.to_numpy() - selected_val_pred)
interval_q90 = float(np.quantile(absolute_residuals, 0.90))

print(f"Frozen 90% residual interval half-width: {interval_q90:.2f} progression points")

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(absolute_residuals, bins=22)
ax.axvline(interval_q90, linestyle="--", label=f"90th percentile = {interval_q90:.1f}")
ax.set_title("Validation absolute forecast errors")
ax.set_xlabel("Absolute error")
ax.set_ylabel("Patients")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Separate high-progression early-warning classifier

The high-risk definition is created from the **training target only**: progression at or above the training-set 75th percentile.

In [ ]:
risk_threshold = float(y_train.quantile(0.75))
y_train_risk = (y_train >= risk_threshold).astype(int)
y_val_risk = (y_val >= risk_threshold).astype(int)
y_test_risk = (y_test >= risk_threshold).astype(int)

logit = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)
logit.fit(X_train_scaled, y_train_risk)

rf_clf = RandomForestClassifier(
    n_estimators=700,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    max_features=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train_risk)

val_prob_logit = logit.predict_proba(X_val_scaled)[:,1]
val_prob_rf = rf_clf.predict_proba(X_val)[:,1]

def classification_metrics(name, y_true, prob, threshold=0.50):
    pred = (prob >= threshold).astype(int)
    return {
        "model": name,
        "ROC_AUC": roc_auc_score(y_true, prob),
        "PR_AUC": average_precision_score(y_true, prob),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0)
    }

classifier_results = pd.DataFrame([
    classification_metrics("Logistic Regression", y_val_risk, val_prob_logit),
    classification_metrics("Random Forest", y_val_risk, val_prob_rf)
]).set_index("model").sort_values("PR_AUC", ascending=False)

display(classifier_results.style.format({
    "ROC_AUC":"{:.4f}",
    "PR_AUC":"{:.4f}",
    "precision":"{:.2%}",
    "recall":"{:.2%}",
    "F1":"{:.4f}"
}))

best_classifier_name = classifier_results.index[0]
print(f"Training-defined high-risk threshold: {risk_threshold:.1f}")
print("Validation-selected classifier:", best_classifier_name)

## 8. Threshold selection prioritising recall

For an early-warning queue, missed high-risk cases matter. The operating threshold is selected on validation data to maximise F1 while requiring at least 70% recall when possible.

In [ ]:
selected_val_prob = val_prob_rf if best_classifier_name == "Random Forest" else val_prob_logit

threshold_grid = np.linspace(0.05, 0.95, 181)
rows = []
for t in threshold_grid:
    pred = (selected_val_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_val_risk, pred, zero_division=0),
        "recall": recall_score(y_val_risk, pred, zero_division=0),
        "F1": f1_score(y_val_risk, pred, zero_division=0)
    })

threshold_table = pd.DataFrame(rows)
eligible = threshold_table.loc[threshold_table["recall"] >= 0.70]
if len(eligible):
    chosen = eligible.sort_values(["F1","precision"], ascending=False).iloc[0]
else:
    chosen = threshold_table.sort_values("F1", ascending=False).iloc[0]

risk_operating_threshold = float(chosen["threshold"])
display(threshold_table.sort_values("F1", ascending=False).head(10))
print(f"Frozen clinical-review threshold: {risk_operating_threshold:.3f}")
print(f"Validation recall at threshold: {chosen['recall']:.2%}")
print(f"Validation precision at threshold: {chosen['precision']:.2%}")

## 9. Untouched final test evaluation

In [ ]:
test_regression_predictions = {
    "Ridge": ridge.predict(X_test_scaled),
    "Random Forest": rf.predict(X_test),
    "HistGradientBoosting": hgb.predict(X_test)
}
test_pred = test_regression_predictions[best_regression_name]

test_regression_result = regression_metrics(best_regression_name, y_test, test_pred)
display(pd.DataFrame([test_regression_result]).set_index("model").style.format({
    "MAE":"{:.2f}",
    "RMSE":"{:.2f}",
    "R2":"{:.4f}"
}))

lower = test_pred - interval_q90
upper = test_pred + interval_q90
interval_coverage = np.mean((y_test.to_numpy() >= lower) & (y_test.to_numpy() <= upper))
print(f"Frozen 90% residual-band test coverage: {interval_coverage:.2%}")

In [ ]:
test_prob_logit = logit.predict_proba(X_test_scaled)[:,1]
test_prob_rf = rf_clf.predict_proba(X_test)[:,1]
test_prob = test_prob_rf if best_classifier_name == "Random Forest" else test_prob_logit
test_pred_risk = (test_prob >= risk_operating_threshold).astype(int)

test_classification_result = classification_metrics(
    best_classifier_name,
    y_test_risk,
    test_prob,
    risk_operating_threshold
)

display(pd.DataFrame([test_classification_result]).set_index("model").style.format({
    "ROC_AUC":"{:.4f}",
    "PR_AUC":"{:.4f}",
    "precision":"{:.2%}",
    "recall":"{:.2%}",
    "F1":"{:.4f}"
}))

print(classification_report(
    y_test_risk,
    test_pred_risk,
    target_names=["Lower progression risk","High progression risk"],
    zero_division=0
))

cm = confusion_matrix(y_test_risk, test_pred_risk)
fig, ax = plt.subplots(figsize=(5.4, 4.4))
im = ax.imshow(cm)
ax.set_title("Final high-progression risk confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0,1], ["Lower","High"])
ax.set_yticks([0,1], ["Lower","High"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 10. Prediction quality visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, test_pred, alpha=0.75)
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
ax.plot(lims, lims, linestyle="--")
ax.set_title("Actual vs predicted one-year disease progression")
ax.set_xlabel("Actual progression")
ax.set_ylabel("Predicted progression")
plt.tight_layout()
plt.show()

precision, recall, _ = precision_recall_curve(y_test_risk, test_prob)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(recall, precision)
ax.set_title("High-progression risk precision-recall curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Explainability with permutation importance

Permutation importance asks how much predictive performance deteriorates when each general clinical signal is disrupted.

In [ ]:
selected_reg_model = {
    "Ridge": ridge,
    "Random Forest": rf,
    "HistGradientBoosting": hgb
}[best_regression_name]

if best_regression_name == "Ridge":
    perm = permutation_importance(
        selected_reg_model, X_test_scaled, y_test,
        n_repeats=50, random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error"
    )
else:
    perm = permutation_importance(
        selected_reg_model, X_test, y_test,
        n_repeats=50, random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error"
    )

importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

display(importance.to_frame("permutation_importance"))

fig, ax = plt.subplots(figsize=(8, 5))
importance.sort_values().plot(kind="barh", ax=ax)
ax.set_title("General clinical drivers of progression forecast")
ax.set_xlabel("Permutation importance")
plt.tight_layout()
plt.show()

## 12. Ranked clinical-review queue

The queue is anonymised with synthetic review IDs. It ranks by predicted progression and high-risk probability.

This is **not** a treatment recommendation; it is an analytical triage demonstration.

In [ ]:
review = X_test.copy().reset_index(drop=True)
review["actual_progression"] = y_test.reset_index(drop=True)
review["predicted_progression"] = test_pred
review["prediction_lower"] = lower
review["prediction_upper"] = upper
review["high_risk_probability"] = test_prob
review["high_risk_flag"] = test_pred_risk

review["review_id"] = [f"CASE-{i+1:03d}" for i in range(len(review))]
review["priority_score"] = (
    0.60 * (review["predicted_progression"] - review["predicted_progression"].min()) /
    (review["predicted_progression"].max() - review["predicted_progression"].min() + 1e-9)
    + 0.40 * review["high_risk_probability"]
)

queue = review.sort_values(
    ["high_risk_flag","priority_score"],
    ascending=[False,False]
)

queue_view = queue[[
    "review_id","predicted_progression","prediction_lower","prediction_upper",
    "high_risk_probability","high_risk_flag",
    "BMI","Average blood pressure","HDL","Blood glucose"
]].head(25)

print("Top 25 analytical review cases:")
display(queue_view.style.format({
    "predicted_progression":"{:.1f}",
    "prediction_lower":"{:.1f}",
    "prediction_upper":"{:.1f}",
    "high_risk_probability":"{:.2%}",
    "BMI":"{:.4f}",
    "Average blood pressure":"{:.4f}",
    "HDL":"{:.4f}",
    "Blood glucose":"{:.4f}"
}))

queue.to_csv("healthcare_progression_review_queue.csv", index=False)
print("Review queue exported.")

## 13. Executive summary

In [ ]:
print(f"""
HEALTHCARE DISEASE-PROGRESSION EARLY-WARNING SUMMARY
----------------------------------------------------
Dataset: scikit-learn diabetes progression dataset
Patients: {len(X):,}
Prediction horizon: one year after baseline
Model inputs: BMI, blood pressure + six serum measures
Demographic predictors used: NONE
Age used: NO
Sex used: NO

Selected regression model: {best_regression_name}
Test MAE: {test_regression_result['MAE']:.2f}
Test RMSE: {test_regression_result['RMSE']:.2f}
Test R²: {test_regression_result['R2']:.4f}
Residual-band test coverage: {interval_coverage:.2%}

Selected high-risk classifier: {best_classifier_name}
Test ROC-AUC: {test_classification_result['ROC_AUC']:.4f}
Test PR-AUC: {test_classification_result['PR_AUC']:.4f}
Test precision: {test_classification_result['precision']:.2%}
Test recall: {test_classification_result['recall']:.2%}
Test F1: {test_classification_result['F1']:.4f}

The project deliberately separates continuous progression forecasting from
a high-risk alert layer, freezes model/threshold choices before the test set,
adds uncertainty bands, explains general clinical drivers and produces an
anonymised analytical review queue.

Educational research demonstration only — not a clinical diagnostic system.
""")

### Production roadmap
- external validation on a substantially larger contemporary cohort;
- patient-disjoint temporal evaluation where repeated records exist;
- prospective calibration and subgroup performance audits;
- missingness-aware longitudinal vitals / laboratory modelling;
- clinically validated risk thresholds and escalation protocols;
- uncertainty-aware abstention for out-of-distribution patients;
- model drift and data-quality monitoring;
- clinician-in-the-loop review before any real-world use.

### Interview defence
**“I intentionally removed age and sex before modelling. I predicted one-year diabetes progression from BMI, blood pressure and serum measurements, selected models only on validation data, froze an uncertainty band and high-risk threshold before final testing, and converted the outputs into an anonymised review queue. I would not present it as a diagnostic tool without external clinical validation.”**